In [ ]:
!pip install \
    "requests>=2.31.0" \
    "numpy>=1.24.0" \
    "sentence-transformers>=2.2.0" \
    "faiss-cpu>=1.7.4" \
    "ipykernel>=6.0.0" \
    "jupyter>=1.0.0"

In [ ]:
# =============================================================================
# CELL 1: IMPORTS AND CONFIGURATION
# =============================================================================
#
# This cell sets up all required libraries and configuration parameters.
# - vLLM URL: The address where your vLLM server is running
# - MODEL: The specific model deployed on your vLLM server
# - DATA_DIR: Folder where all persistent JSON/FAISS data is stored
#
#  HOW TO CONFIGURE:
#  -----------------
#  Change VLLM_URL and MODEL below to match your vLLM deployment.
#  If vLLM is on another machine, use its IP/hostname.
# =============================================================================

import os
import json
import requests
import numpy as np
from datetime import datetime

# ---------------------------------------------------------------------------
# vLLM Configuration â€” Customize these for your environment
# ---------------------------------------------------------------------------
VLLM_URL = "http://localhost:8000"               # Base URL of your vLLM server
MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"    # Model deployed on vLLM
TEMPERATURE = 0.2                                 # Lower = more deterministic

# ---------------------------------------------------------------------------
# File Paths â€” All persistent storage lives under data/
# ---------------------------------------------------------------------------
DATA_DIR = "data"
INCIDENT_FILE = os.path.join(DATA_DIR, "incidents.json")
HISTORY_FILE = os.path.join(DATA_DIR, "analysis_history.json")
FAISS_INDEX_FILE = os.path.join(DATA_DIR, "faiss.index")

# ---------------------------------------------------------------------------
# Embedding Model Settings
# ---------------------------------------------------------------------------
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # Sentence-Transformers model name
EMBEDDING_DIM = 384                      # Output dimension of the embedding model

print("[CONFIG] Configuration loaded.")
print(f"[CONFIG] vLLM Endpoint : {VLLM_URL}")
print(f"[CONFIG] Model         : {MODEL}")
print(f"[CONFIG] Data Directory : {os.path.abspath(DATA_DIR)}")


In [ ]:
# =============================================================================
# CELL 2: STORAGE SETUP
# =============================================================================
#
# Ensures the data/ directory and JSON storage files exist before any agent runs.
# - incidents.json        : Database of known infrastructure incidents
# - analysis_history.json : History of all pipeline runs with full outputs
#
# Both files start as empty arrays [] and grow over time as the system learns.
# =============================================================================

os.makedirs(DATA_DIR, exist_ok=True)

# --- Initialize incidents.json if missing ---
if not os.path.exists(INCIDENT_FILE):
    with open(INCIDENT_FILE, "w", encoding="utf-8") as f:
        json.dump([], f)
    print(f"[INIT] Created {INCIDENT_FILE}")

# --- Initialize analysis_history.json if missing ---
if not os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump([], f)
    print(f"[INIT] Created {HISTORY_FILE}")

print("[INIT] Storage directories and files are ready.")


In [ ]:
# =============================================================================
# CELL 3: FAISS + SENTENCE-TRANSFORMERS EMBEDDING ENGINE
# =============================================================================
#
# This cell loads the Sentence-Transformers model and provides functions for
# building and querying a FAISS index. The index enables semantic search over
# historic incidents so the system can find similar past failures.
#
# Key Concepts:
# - Sentence-Transformers converts text to dense vectors (embeddings)
# - FAISS IndexFlatL2 uses L2 (Euclidean) distance for similarity
# - The index is rebuilt on each query to stay in sync with JSON storage
# =============================================================================

import faiss
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Load the embedding model (loaded once at notebook startup)
# ---------------------------------------------------------------------------
print(f"[INIT] Loading embedding model: {EMBEDDING_MODEL} ...")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f"[INIT] Embedding model loaded. Vector dimension: {EMBEDDING_DIM}")


# ---------------------------------------------------------------------------
def build_faiss_index(incidents_list):
    """
    Build a FAISS index from the provided list of incidents.

    Each incident is converted to a flat search string and embedded.
    The resulting vectors are indexed with FAISS IndexFlatL2.

    Parameters
    ----------
    incidents_list : list[dict]
        List of incident dicts with keys: change, incident, severity, impact

    Returns
    -------
    tuple : (faiss.Index, list[str])
        The built FAISS index and corresponding text representations
    """
    # Convert each incident to a searchable text blob
    texts = []
    for inc in incidents_list:
        text = f"Change: {inc['change']} | Incident: {inc['incident']} | Severity: {inc['severity']} | Impact: {inc['impact']}"
        texts.append(text)

    if len(texts) == 0:
        print("[FAISS] No incidents to index. Returning empty index.")
        return faiss.IndexFlatL2(EMBEDDING_DIM), []

    # Batch-encode all incident texts to vectors
    embeddings = embedder.encode(texts, show_progress_bar=False)

    # Build FAISS index using L2 distance
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings).astype(np.float32))

    # Persist the index to disk for potential reuse
    faiss.write_index(index, FAISS_INDEX_FILE)

    print(f"[FAISS] Index built with {len(texts)} incidents. Dimension: {dimension}")
    return index, texts


# ---------------------------------------------------------------------------
def retrieve_incidents(query_text, k=5):
    """
    Retrieve the top-k most semantically similar incidents to the query.

    This is the core retrieval function used by the Incident Retrieval Agent.
    It loads the latest incidents from disk, builds/rebuilds the FAISS index,
    embeds the query, and searches for the nearest neighbors.

    Parameters
    ----------
    query_text : str
        The change request text to search by
    k : int
        Number of similar incidents to return (default: 5)

    Returns
    -------
    str
        Formatted string of top-k incidents with their similarity distances
    """
    # Load the latest incidents from disk
    with open(INCIDENT_FILE, "r", encoding="utf-8") as f:
        incidents = json.load(f)

    if len(incidents) == 0:
        return "No incidents in database to compare."

    # Build index fresh (ensures sync with latest incidents)
    index, texts = build_faiss_index(incidents)

    # Embed the query text
    query_embedding = embedder.encode([query_text])

    # Search FAISS for k nearest neighbors
    distances, indices = index.search(
        np.array(query_embedding).astype(np.float32),
        min(k, len(incidents))
    )

    # Format results as readable text
    results = []
    for i, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        inc = incidents[idx]
        results.append(
            f"Rank {i+1} (Distance: {distances[0][i]:.4f})\n"
            f"  Change   : {inc['change']}\n"
            f"  Incident : {inc['incident']}\n"
            f"  Severity : {inc['severity']}\n"
            f"  Impact   : {inc['impact']}\n"
        )

    return "\n".join(results) if results else "No similar incidents found."


# ---------------------------------------------------------------------------
# Build the initial index on startup
# ---------------------------------------------------------------------------
print("[INIT] Building initial FAISS index...")
with open(INCIDENT_FILE, "r", encoding="utf-8") as f:
    initial_incidents = json.load(f)
if initial_incidents:
    build_faiss_index(initial_incidents)
print("[INIT] FAISS embedding engine is ready.")


In [ ]:
# =============================================================================
# CELL 4: SEED INCIDENTS
# =============================================================================
#
# Loads existing incidents from persistent storage. If this is the first run
# and the database is empty, seeds it with 2 default infrastructure incidents.
# These seed incidents give the FAISS index initial data to search against.
#
# Over time, the learn_from_run() function (Cell 14) adds new incidents here
# automatically after every pipeline execution.
# =============================================================================

with open(INCIDENT_FILE, "r", encoding="utf-8") as f:
    incidents = json.load(f)

# --- If this is the first run, seed with default incidents ---
if len(incidents) == 0:
    incidents = [
        {
            "change": "database migration",
            "incident": "table lock caused outage",
            "severity": "high",
            "impact": "service unavailable for 15 minutes"
        },
        {
            "change": "redis upgrade",
            "incident": "cache incompatibility",
            "severity": "medium",
            "impact": "latency increase by 200ms"
        }
    ]
    with open(INCIDENT_FILE, "w", encoding="utf-8") as f:
        json.dump(incidents, f, indent=2, ensure_ascii=False)
    print("[SEED] Default incidents written to storage.")
else:
    print(f"[LOAD] Loaded {len(incidents)} existing incidents from storage.")

# --- Display current incident database for visibility ---
print("\nCurrent Incident Database:")
print("=" * 60)
for i, inc in enumerate(incidents, 1):
    print(f"{i}. [{inc['severity'].upper()}] {inc['change']} -> {inc['incident']}")
print("=" * 60)


In [ ]:
# =============================================================================
# CELL 5: STORAGE FUNCTIONS
# =============================================================================
#
# These two functions are the persistence layer of the system. Every pipeline
# run calls save_analysis() to archive the full output, and learn_from_run()
# calls save_incident() to grow the incident database.
# =============================================================================


def save_incident(change, incident, severity, impact):
    """
    Append a new incident record to the JSON database.

    This function is called automatically by learn_from_run() (Cell 14)
    after every pipeline execution. Each call grows the incident database,
    making future FAISS retrievals more informed.

    Parameters
    ----------
    change   : str â€” The infrastructure change that caused the incident
    incident : str â€” Description of what went wrong
    severity : str â€” One of: high, medium, low
    impact   : str â€” Business or technical impact description
    """
    # Load existing incidents from disk
    with open(INCIDENT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Append the new incident record
    data.append({
        "change": change,
        "incident": incident,
        "severity": severity,
        "impact": impact
    })

    # Write the updated list back to disk
    with open(INCIDENT_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"[STORAGE] Incident saved: {change} -> {incident}")


def save_analysis(change, plan, risk, solution, rollout, final_output):
    """
    Persist a complete analysis pipeline run to the history file.

    Every call to run_pipeline() ends with this function, storing the
    full output of all six agents. The Memory Agent (Cell 7) loads these
    records in future runs to provide institutional context.

    Parameters
    ----------
    change       : str â€” The original change request text
    plan         : str â€” Planner agent output
    risk         : str â€” Risk agent output
    solution     : str â€” Solution agent output
    rollout      : str â€” Rollout agent output
    final_output : str â€” Refiner agent executive report
    """
    # Load existing history from disk
    with open(HISTORY_FILE, "r", encoding="utf-8") as f:
        history = json.load(f)

    # Append the new analysis record with a timestamp
    history.append({
        "timestamp": str(datetime.now()),
        "change": change,
        "planner": plan,
        "risk": risk,
        "solution": solution,
        "rollout": rollout,
        "final_output": final_output
    })

    # Write the updated history back to disk
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)

    print(f"[STORAGE] Analysis saved at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# =============================================================================
# CELL 6: LLM CORE â€” vLLM Communication Layer
# =============================================================================
#
# ask_llm():     Direct interface to the vLLM API (OpenAI-compatible)
# ask_agent():   Role-prompting wrapper that enables multi-agent behavior
#
# The design philosophy:
# - ask_llm() is the low-level transport â€” it handles HTTP, retries, errors
# - ask_agent() is the high-level abstraction â€” it makes the model act as
#   any expert role we need (Planner, SRE, Architect, etc.)
# =============================================================================


def ask_llm(prompt, temperature=None):
    """
    Send a prompt to the vLLM API and return the generated response.

    Uses the OpenAI-compatible /v1/chat/completions endpoint exposed by vLLM.
    This allows us to use any model deployed on vLLM without additional SDKs.

    Parameters
    ----------
    prompt      : str â€” The prompt text to send to the model
    temperature : float or None â€” Override the default temperature (optional)

    Returns
    -------
    str â€” The model's generated text response, or an error message

    Error Handling
    --------------
    - ConnectionError: vLLM server is not running or unreachable
    - Timeout: Request took longer than 120 seconds
    - Generic Exception: Catches JSON parsing, HTTP errors, etc.
    """
    # Use default temperature unless overridden
    temp = temperature if temperature is not None else TEMPERATURE

    # Build the request payload following OpenAI chat format
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temp,
        "max_tokens": 2048,
        "top_p": 0.95
    }

    try:
        # POST to the vLLM chat completions endpoint
        response = requests.post(
            f"{VLLM_URL}/v1/chat/completions",
            json=payload,
            timeout=120  # 2-minute timeout for long generations
        )
        response.raise_for_status()  # Raise on HTTP 4xx/5xx

        # Extract the assistant's message from the response
        result = response.json()
        return result["choices"][0]["message"]["content"]

    except requests.exceptions.ConnectionError:
        return "[ERROR] Cannot connect to vLLM server. Ensure it is running at {VLLM_URL}."
    except requests.exceptions.Timeout:
        return "[ERROR] vLLM request timed out after 120 seconds."
    except Exception as e:
        return f"[ERROR] LLM call failed: {str(e)}"


def ask_agent(role, task):
    """
    Act as a specialized agent with a given role to perform a task.

    This is the core multi-agent mechanism. By prepending a role description
    to the prompt, we get the LLM to adopt different expert personas without
    needing separate model instances or fine-tuning.

    Parameters
    ----------
    role : str â€” The expert role to adopt (e.g., "Infrastructure Planner")
    task : str â€” The task description or context for the agent to process

    Returns
    -------
    str â€” The agent's response in the adopted role

    Example
    -------
    >>> plan = ask_agent("Infrastructure Planner", "Plan a Redis upgrade")
    >>> risk = ask_agent("Risk Expert", plan)
    """
    prompt = f"""
You are acting as: {role}

Your task:
{task}

Respond in a concise, technical manner. Use markdown formatting.
"""
    return ask_llm(prompt)


print("[LLM] Core LLM functions initialized.")
print(f"[LLM] Endpoint : {VLLM_URL}")
print(f"[LLM] Model    : {MODEL}")


In [ ]:
# =============================================================================
# CELL 7: MEMORY AGENT
# =============================================================================
#
# Role: Institutional Memory of the Infrastructure Team
#
# The Memory Agent loads past analysis records and extracts relevant lessons.
# It uses a sliding window of the last 10 analyses to keep context manageable.
#
# Why this matters:
# - Without memory, every run is a fresh start with no learning
# - With memory, the system gets smarter over time as the history grows
# - It catches patterns like "every time we upgrade X, Y breaks"
# =============================================================================


def memory_agent(change):
    """
    Load past analyses and summarize relevant lessons for the current change.

    Parameters
    ----------
    change : str â€” The current change request being analyzed

    Returns
    -------
    str â€” Markdown summary of past lessons and relevant history
    """
    # Load analysis history from persistent storage
    with open(HISTORY_FILE, "r", encoding="utf-8") as f:
        history = json.load(f)

    # If no history exists, inform the pipeline
    if len(history) == 0:
        return ("**Memory Agent:** No previous history available. "
                "This is the first analysis run.")

    # Use the last 10 analyses as a sliding context window
    context = history[-10:]

    # Ask the LLM to analyze past records for relevant patterns
    prompt = f"""
You are the Memory Agent â€” the institutional memory of the infrastructure team.

## Current Change Request
{change}

## Past Analyses (Last {len(context)} Records)
{json.dumps(context, indent=2)}

## Task
Review the past analyses above and identify:

1. **Similar Changes** â€” Have we analyzed similar changes before?
2. **Previous Failures** â€” What went wrong in past changes?
3. **Successful Rollbacks** â€” What rollback strategies worked well?
4. **Useful Lessons** â€” What should we keep in mind for this change?

Return your findings in markdown format with clear sections.
"""
    return ask_llm(prompt)


In [ ]:
# =============================================================================
# CELL 8: PLANNER AGENT
# =============================================================================
#
# Role: Infrastructure Planner
#
# Generates a detailed, actionable change plan with:
# - Pre-requisites and preparation steps
# - Ordered execution steps with validation at each stage
# - Dependencies and resource requirements
# - Timeline estimates and success criteria
# =============================================================================


def planner_agent(change):
    """
    Generate a technical infrastructure change plan.

    Parameters
    ----------
    change : str â€” The change request description

    Returns
    -------
    str â€” Markdown formatted change plan
    """
    return ask_agent(
        "Infrastructure Planner â€” Expert in infrastructure changes, migrations, and upgrades",
        f"""
Create a detailed step-by-step infrastructure change plan for:

{change}

Include:
1. Pre-requisites and preparation steps
2. Step-by-step execution plan
3. Validation checkpoints at each step
4. Dependencies and ordering constraints
5. Estimated duration for each step
6. Success criteria for completion

Format as a markdown checklist.
"""
    )


In [ ]:
# =============================================================================
# CELL 9: INCIDENT RETRIEVAL AGENT
# =============================================================================
#
# Role: Incident Analysis Specialist
#
# This agent combines two steps:
# 1. FAISS semantic search to find similar past incidents
# 2. LLM analysis to extract patterns, risk indicators, and impacts
#
# This is the RAG (Retrieval-Augmented Generation) component of the pipeline.
# It grounds the risk assessment in real historical data rather than relying
# solely on the LLM's parametric knowledge.
# =============================================================================


def retrieval_agent(change):
    """
    Search past incidents using FAISS and analyze the retrieved patterns.

    Parameters
    ----------
    change : str â€” The change request to search by

    Returns
    -------
    str â€” Markdown analysis of retrieved incidents and failure patterns
    """
    # Step 1: Use FAISS to find semantically similar incidents
    similar_incidents = retrieve_incidents(change, k=5)

    # Step 2: Ask the LLM to analyze the retrieved incidents
    prompt = f"""
You are the Incident Analysis Agent, specialized in identifying failure patterns from historical data.

## Current Change Request
{change}

## Similar Past Incidents (Retrieved via FAISS semantic search)
{similar_incidents}

## Task
Analyze the retrieved incidents and produce a structured summary:

1. **Common Failure Patterns** â€” What tends to go wrong with changes like this?
2. **Risk Indicators** â€” What warning signs should we watch for?
3. **Historical Impacts** â€” What was the blast radius of past incidents?
4. **Severity Distribution** â€” How severe were past incidents?

Return your analysis in markdown format.
"""
    return ask_llm(prompt)


In [ ]:
# =============================================================================
# CELL 10: RISK AGENT
# =============================================================================
#
# Role: Risk Assessment Expert
#
# Evaluates risk across multiple dimensions:
# - Service disruption likelihood and impact
# - Data loss or corruption risk
# - Performance degradation
# - Security implications
# - Rollback complexity
#
# Output includes an overall risk score (HIGH/MEDIUM/LOW) and specific
# risk items that the Solution Agent will address.
# =============================================================================


def risk_agent(change_with_context):
    """
    Assess the risk of the proposed infrastructure change.

    Parameters
    ----------
    change_with_context : str â€” The change request plus retrieved incident context

    Returns
    -------
    str â€” Markdown risk assessment with overall score and breakdown
    """
    return ask_agent(
        "Risk Assessment Expert â€” Senior SRE specializing in change risk analysis",
        f"""
Evaluate the risk of the following infrastructure change:

{change_with_context}

Provide:
1. **Overall Risk Score** â€” HIGH / MEDIUM / LOW with clear justification
2. **Risk Breakdown by Category**:
   - Service disruption risk
   - Data loss or corruption risk
   - Performance degradation risk
   - Security risk
   - Rollback complexity
3. **Critical Risk Items** â€” Top 3 risks that must be addressed
4. **Risk Mitigation Suggestions** â€” Quick wins to reduce risk level

Be specific and technical. Reference real infrastructure failure patterns.
"""
    )


In [ ]:
# =============================================================================
# CELL 11: SOLUTION AGENT
# =============================================================================
#
# Role: Principal Site Reliability Engineer
#
# Generates a comprehensive mitigation plan addressing the risks identified
# by the Risk Agent. Covers the full lifecycle: before, during, and after
# the change is applied.
# =============================================================================


def solution_agent(change, risk):
    """
    Generate preventive measures and mitigations for identified risks.

    Parameters
    ----------
    change : str â€” The original change request
    risk   : str â€” The risk assessment output from risk_agent()

    Returns
    -------
    str â€” Markdown mitigation plan with actionable steps
    """
    return ask_agent(
        "Principal Site Reliability Engineer â€” Expert in production safety and incident prevention",
        f"""
## Change Request
{change}

## Risk Assessment
{risk}

## Task
Generate a comprehensive mitigation plan:

1. **Preventive Actions** â€” What to do before the change to reduce risk
2. **Required Testing** â€” What tests must pass before proceeding to production
3. **Monitoring Rules** â€” What metrics, logs, and alerts to configure
4. **Auto-healing Options** â€” What can be automated to reduce manual intervention
5. **Fallback Design** â€” How to safely return to the previous state if needed

Be practical and specific. Prioritize actions by risk reduction impact.
"""
    )


In [ ]:
# =============================================================================
# CELL 12: ROLLOUT AGENT
# =============================================================================
#
# Role: Release Engineering Expert
#
# Designs a production-safe rollout strategy with:
# - Deployment methodology (canary, blue-green, rolling)
# - Phased rollout with health check gates between phases
# - Automated rollback triggers based on metrics
# - Manual rollback procedure as a safety net
# - Communication plan for stakeholders
# =============================================================================


def rollout_agent(change, risk):
    """
    Design a safe rollout strategy with comprehensive rollback procedures.

    Parameters
    ----------
    change : str â€” The change request
    risk   : str â€” The risk assessment output

    Returns
    -------
    str â€” Markdown rollout and rollback plan
    """
    return ask_agent(
        "Release Engineering Expert â€” Specialist in safe production deployments and rollbacks",
        f"""
## Change Request
{change}

## Risk Assessment
{risk}

## Task
Design a production rollout plan:

1. **Deployment Strategy** â€” Canary, blue-green, or rolling update? Justify choice.
2. **Phased Rollout Plan** â€” Step-by-step with gating criteria for each phase
3. **Health Check Gates** â€” What metrics determine a phase is healthy?
4. **Automated Rollback Triggers** â€” When should auto-rollback fire?
5. **Manual Rollback Procedure** â€” Step-by-step rollback instructions
6. **Communication Plan** â€” Who to notify and at which stages

Format as an actionable runbook that an on-call engineer could follow.
"""
    )


In [ ]:
# =============================================================================
# CELL 13: REFINER AGENT â€” Executive Report
# =============================================================================
#
# Role: Chief Infrastructure Architect
#
# This agent synthesizes ALL prior agent outputs into one final report.
# It is the convergence point of the entire multi-agent pipeline.
#
# Report sections:
# - Executive Summary
# - Risk Score (HIGH/MEDIUM/LOW)
# - Predicted Incidents (from historical patterns)
# - Business Impact
# - Mitigation Strategy
# - Rollout Plan
# - Rollback Plan
# - Final Recommendation (APPROVE / APPROVE WITH CAUTION / REJECT)
# =============================================================================


def refiner_agent(change, memory, plan, risks, solution, rollout):
    """
    Combine all agent outputs into a single executive report.

    This is the final step in the pipeline before persistence and learning.
    It produces a professional-grade report suitable for stakeholders.

    Parameters
    ----------
    change   : str â€” Original change request
    memory   : str â€” Memory Agent output (past lessons)
    plan     : str â€” Planner Agent output (change plan)
    risks    : str â€” Risk Agent output (risk assessment)
    solution : str â€” Solution Agent output (mitigations)
    rollout  : str â€” Rollout Agent output (deployment strategy)

    Returns
    -------
    str â€” Executive report in markdown with final recommendation
    """
    prompt = f"""
You are the Chief Infrastructure Architect â€” the final decision-maker for all infrastructure changes.

Review all input below and produce a single, cohesive executive report.

## Infrastructure Change Request
{change}

## Institutional Memory (Past Lessons Learned)
{memory}

## Change Plan
{plan}

## Risk Assessment
{risks}

## Mitigation Solutions
{solution}

## Rollout Strategy
{rollout}

## Instructions
Create a professional executive report with these exact sections:

# Executive Summary

# Risk Score

# Predicted Incidents

# Business Impact

# Mitigation Strategy

# Rollout Plan

# Rollback Plan

# Final Recommendation
**APPROVE** â€” Safe to proceed
**APPROVE WITH CAUTION** â€” Proceed with specific conditions listed
**REJECT** â€” Do not proceed, with reasons why

Use professional markdown formatting. Be decisive and specific.
"""
    return ask_llm(prompt)


In [ ]:
# =============================================================================
# CELL 14: LEARNING MECHANISM
# =============================================================================
#
# This function closes the learning loop:
#   Run -> Analyze -> Extract Incidents -> Store -> Future Runs Get Smarter
#
# After each pipeline run, we ask the LLM to read the final report and
# extract structured incident records. These are appended to the incident
# database, growing the knowledge base for future FAISS searches.
#
# This means the system gets better with every use â€” even without manual
# data entry of new incidents.
# =============================================================================


def learn_from_run(change, final_output):
    """
    Extract incident knowledge from the final report and persist it.

    This is the continuous learning mechanism. After each pipeline execution,
    we parse the final report for any described incidents or failure modes
    and add them to the incident database for future reference.

    Parameters
    ----------
    change       : str â€” The original change request
    final_output : str â€” The refiner agent's executive report
    """
    # Ask the LLM to extract structured incident data from the report
    prompt = f"""
Extract any infrastructure incident knowledge from the analysis below.

## Change Request
{change}

## Analysis Report
{final_output}

## Task
If the report describes potential incidents or failure scenarios, return them as a JSON array.
Each incident must have exactly these fields:
- "change": The change that could cause the incident
- "incident": Description of what could go wrong
- "severity": "high", "medium", or "low"
- "impact": Business or technical impact description

Return ONLY a valid JSON array. Example:
[
  {{
    "change": "database migration",
    "incident": "table lock causing outage",
    "severity": "high",
    "impact": "service unavailable"
  }}
]

If no incidents are described, return an empty array: []
"""

    result = ask_llm(prompt)

    # Parse the JSON response with robust error handling
    # LLMs sometimes wrap JSON in markdown code blocks
    try:
        # Try to find JSON array in the response (handle markdown wrapping)
        json_start = result.find("[")
        json_end = result.rfind("]") + 1
        if json_start >= 0 and json_end > json_start:
            json_str = result[json_start:json_end]
            new_incidents = json.loads(json_str)
        else:
            new_incidents = json.loads(result)

        # Append any extracted incidents to the database
        if new_incidents and len(new_incidents) > 0:
            with open(INCIDENT_FILE, "r", encoding="utf-8") as f:
                existing = json.load(f)

            existing.extend(new_incidents)

            with open(INCIDENT_FILE, "w", encoding="utf-8") as f:
                json.dump(existing, f, indent=2, ensure_ascii=False)

            print(f"[LEARN] Extracted and stored {len(new_incidents)} new incident(s) from this run.")
        else:
            print("[LEARN] No new incidents extracted from this run.")

    except (json.JSONDecodeError, ValueError, Exception) as e:
        # If parsing fails, log the error and continue silently
        # The pipeline should not crash because of a failed extraction
        print(f"[LEARN] Could not extract incidents from report: {e}")


In [ ]:
# =============================================================================
# CELL 15: FULL PIPELINE
# =============================================================================
#
# This is the main orchestrator that runs all 7 agents in sequence,
# then persists the results and extracts new knowledge.
#
# Pipeline Flow:
#   change_request
#     -> memory_agent()      : Load institutional memory
#     -> planner_agent()     : Generate change plan
#     -> retrieval_agent()   : FAISS search for similar incidents
#     -> risk_agent()        : Assess risk with incident context
#     -> solution_agent()    : Generate mitigations
#     -> rollout_agent()     : Design rollout strategy
#     -> refiner_agent()     : Produce executive report
#     -> save_analysis()     : Persist to history
#     -> learn_from_run()    : Extract new incidents
#     -> return final_report
# =============================================================================


def run_pipeline(change_request):
    """
    Execute the complete multi-agent infrastructure change risk pipeline.

    This function orchestrates all 7 agents, persistence, and learning
    in the correct order. Each agent's output feeds into the next.

    Parameters
    ----------
    change_request : str â€” The infrastructure change request to analyze

    Returns
    -------
    str â€” The final executive report with APPROVE/REJECT recommendation
    """
    print("=" * 80)
    print("INFRASTRUCTURE CHANGE RISK ANALYSIS PIPELINE")
    print("=" * 80)
    print(f"Change Request: {change_request.strip()[:100]}...\n")

    # --------------------------------------------------------------------
    # STEP 1: Memory Agent â€” Load institutional memory
    # --------------------------------------------------------------------
    print("[1/7] Memory Agent â€” Loading past analyses...")
    memory = memory_agent(change_request)
    print("[1/7] Memory Agent complete.\n")

    # --------------------------------------------------------------------
    # STEP 2: Planner Agent â€” Generate change plan
    # --------------------------------------------------------------------
    print("[2/7] Planner Agent â€” Creating change plan...")
    plan = planner_agent(change_request)
    print("[2/7] Planner Agent complete.\n")

    # --------------------------------------------------------------------
    # STEP 3: Incident Retrieval Agent â€” FAISS search
    # --------------------------------------------------------------------
    print("[3/7] Incident Retrieval Agent â€” Searching past incidents...")
    retrieved = retrieval_agent(change_request)
    print("[3/7] Incident Retrieval Agent complete.\n")

    # --------------------------------------------------------------------
    # STEP 4: Risk Agent â€” Assess risk with incident context
    # --------------------------------------------------------------------
    print("[4/7] Risk Agent â€” Assessing risk...")
    risk_context = f"Change Request:\n{change_request}\n\nRetrieved Incidents:\n{retrieved}"
    risk = risk_agent(risk_context)
    print("[4/7] Risk Agent complete.\n")

    # --------------------------------------------------------------------
    # STEP 5: Solution Agent â€” Generate mitigations
    # --------------------------------------------------------------------
    print("[5/7] Solution Agent â€” Generating mitigations...")
    solution = solution_agent(change_request, risk)
    print("[5/7] Solution Agent complete.\n")

    # --------------------------------------------------------------------
    # STEP 6: Rollout Agent â€” Design rollout strategy
    # --------------------------------------------------------------------
    print("[6/7] Rollout Agent â€” Designing rollout strategy...")
    rollout = rollout_agent(change_request, risk)
    print("[6/7] Rollout Agent complete.\n")

    # --------------------------------------------------------------------
    # STEP 7: Refiner Agent â€” Executive report
    # --------------------------------------------------------------------
    print("[7/7] Refiner Agent â€” Producing executive report...")
    final_report = refiner_agent(
        change_request,
        memory,
        plan,
        risk,
        solution,
        rollout
    )
    print("[7/7] Refiner Agent complete.\n")

    # --------------------------------------------------------------------
    # PERSIST: Save this analysis to history for future memory
    # --------------------------------------------------------------------
    print("[SAVE] Saving analysis to history...")
    save_analysis(change_request, plan, risk, solution, rollout, final_report)

    # --------------------------------------------------------------------
    # LEARN: Extract new incidents from this run
    # --------------------------------------------------------------------
    print("[LEARN] Extracting knowledge from this run...")
    learn_from_run(change_request, final_report)

    # --------------------------------------------------------------------
    # DONE
    # --------------------------------------------------------------------
    print("=" * 80)
    print("PIPELINE COMPLETE")
    print("=" * 80)

    return final_report


In [ ]:
# =============================================================================
# CELL 16: DEMO â€” Run the Pipeline
# =============================================================================
#
# This is the entry point for running the full analysis.
# Simply edit the change_request variable and execute this cell.
#
# The pipeline will:
#   1. Load institutional memory from past runs
#   2. Generate a detailed change plan
#   3. Search for similar past incidents via FAISS
#   4. Assess risk across multiple dimensions
#   5. Generate mitigation strategies
#   6. Design a rollout and rollback plan
#   7. Produce an executive report with final recommendation
#   8. Save everything to persistent storage
#   9. Extract and store any new incident knowledge
# =============================================================================

# --- Define your infrastructure change request ---
change_request = """
Upgrade Redis 6.2 to Redis 7
across production clusters.

Details:
- 5 production clusters (US-East, US-West, EU, APAC, ME)
- 120 Redis nodes total
- Requires config changes for new ACL format
- New features: sharded pub/sub, functions, client-eviction
- Rollback: downgrade is not supported â€” data must be migrated
"""

# --- Run the full pipeline ---
final_report = run_pipeline(change_request)

# --- Display the final executive report ---
from IPython.display import display, Markdown
display(Markdown(final_report))
